## El método de MonteCarlo WaveFunction

Es un método para simular un sistema pequeño acoplado a un reservorio grande, es decir un sistema cuantico abierto, usando funciones de onda (WF) en vez de el operador de densidad podemos disminuir el numero de componentes que se necesita calcular de $N^2 \rightarrow N$, esto mediante un hamiltoniano no hermitico y la evolucion consideramos saltos cuanticos aleatorios que originan las fluctuaciones y disipacion del sistema. Es equivalente a la ecuacion de Lindblad (Regimen Markoviano) tomando el promedio de las 'trayectorias' de la WF.

$$
\dot \rho_s = i [\rho_s, H_s] + \mathcal{L}_{\small relax}(\rho_s)
$$
Donde $\mathcal{L}_{\small relax}(\rho_s)$ es el superop. de relajacion, en el paper de Molmer utiliza ops. de relajacion tipo eq. de Lindblad. El método consiste en 2 pasos

- Evolucion con Hamiltoniano no hermitico y los saltos cuanticos aleatorios
- Renormalizacion de la WF

Consideramos el sig. Hamiltoniano, notar que $H\neq H^\dagger$ y suponemos que en un tiempo t (inicial) se encuentra en un estado puro la WF $\ket {\phi(t)}$
$$
H = H_s - \frac{i}{2} \sum C_m^\dagger C_m
$$
Entonces su evolucion para dt pequeños
$$
\ket {\phi^1(t+dt)} = \left( \mathbb I - i H dt\right)\ket \phi
$$

In [ ]:
import numpy as np
from qutip import *
import sys 
np.set_printoptions(
    threshold=np.inf,      # Never truncate rows/columns
    linewidth=150,         # Max characters per line before wrapping
    precision=4,           # Number of decimal places
    suppress=True          # Suppress scientific notation (e.g., 1e-16 -> 0)
)
import matplotlib.pyplot as plt

In [ ]:


def MCWF_evolution(phi_0, H_S, L_ops, t_final, dt):
    """
    Evolución Monte Carlo Wave Function (MCWF).

    Parameters
    ----------
    phi_0   : Estado inicial ket phi(0).
    H_S     : Hamiltoniano del sistema pequeño.
    L_ops   : Lista de operadores de salto L_m.
    t_final : Tiempo final.
    dt      : Paso temporal.

    Returns
    -------
    resultados : Estados ket phi(t + dt) a cada paso temporal.
    """

    phi = phi_0.copy()
    resultados = []

    t = 0.0

    # Hamiltoniano no hermitico
    H = H_S.copy()

    for L in L_ops:
        H += -1j / 2 * (L.dag() @ L) # Suma L_m^dag L_m

    while t < t_final:

    # 1. Evolución 
        I = qeye(H_S.shape[0])
        phi_1 = (I - 1j * H * dt) @ phi

        # Probabilidades de salto
        delta_p_m = []

        for L in L_ops:
            # dt \bra{phi} L_m^dag L_m \ket{phi}
            v = (phi.dag() @ L.dag() @ L @ phi).real
            dp = dt*v                        # Calcula dp_m para todos los L
            delta_p_m.append(dp)             # Para cada m lo mete a esta lista, individual

        delta_p = sum(delta_p_m)

        # 2. Posible salto cuantico 
        epsilon = np.random.random()

        if epsilon > delta_p: #No ocurre
            #El estado se normaliza como siempre
            phi = phi_1 / phi_1.norm()
        else: # Si ocurre 
            
            probs = np.array(delta_p_m) / delta_p
            m = np.random.choice(len(L_ops), p=probs)  # Toma cualquier (aleatoriamente) L ops
            L = L_ops[m]

            # Cambia el estado y su norma
            norma = np.sqrt(delta_p_m[m] / dt)
            phi = L @ phi / norma

        # Guardar estado 
        resultados.append(phi.copy())

        t += dt # aumenta el t

    return  np.array(resultados)



In [ ]:
alpha = 0.1 - 0.5j

beta = np.sqrt(complex(1 - alpha * alpha.conjugate()))

print(beta * beta.conjugate())
print(alpha * alpha.conjugate())

# Sistema de 2 niveles del oscilador armonico
Acoplado a un baño de osciladores en su estado base (T=0 K) (Acoplamiento tipo emisión espontanea)

In [ ]:
"""
two-level system of a harmonic oscillator with a spontaneous-emission like
coupling , i.e., a coupling to a bath of harmonic oscillators in its ground state (zero temperature). Molmer paper eq3
"""
N = 100# Espacio de Hilbert
ket0 = basis(N, 0) 
ket1 = basis(N,1)

# Parametros 
alpha = 0.1 - 0.5j
if np.abs(alpha) > 1:
    print('el modulo de alpha debe ser menor a 1')
    sys.exit()
else:
    beta = np.sqrt(complex(1 - alpha * alpha.conjugate()))
Gamma = 1 
omega_0 = 1
# Estado inicial
phi_0 = coherent(N, 1)


b = destroy(N)
I = qeye(N)
H_s = omega_0 * (b.dag() * b)
L_ops = [np.sqrt(Gamma) * b]

N_traj = [5, 20, 40, 60]

tray_n = []

for N in N_traj:

    trayectorias = []
    
    for i in range(N):

        trayectoria = MCWF_evolution(
            phi_0,
            H_s,
            L_ops,
            t_final=5,
            dt=0.005
        )

        trayectorias.append(trayectoria)

    tray_n.append(np.array(trayectorias))

print('finished')

In [ ]:
"""
Evolucion exacta de la ecuacion de Linblad para comparar
"""
t = np.arange(0, 5+0.005, 0.005)
rho_0 = phi_0*phi_0.dag()
n_op = [b.dag()*b]
evo1 = mesolve(H_s , rho_0, t, L_ops, e_ops=n_op)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for n in range(len(N_traj)):

    promedio = np.mean(
        [
            np.array([
                expect(n_op, phi)
                for phi in tray
            ])
            for tray in tray_n[n]
        ],
        axis=0
    )

    plt.plot(
        t,
        promedio,
        label=fr"$N={N_traj[n]}$"
    )
plt.plot(t, evo1.expect[0], c='black', label = 'Linblad eq.')

plt.title(r'Promedio de $\langle n \rangle$ para cada  N trayectorias ' + '\n y la sol. numerica')
plt.xlabel(r"$t$")
plt.ylabel(r"$\langle n \rangle$")
plt.legend()
plt.grid()

plt.savefig('HO_twolevels.pdf')
plt.show()



In [ ]:

X = np.sqrt(2)*(b.dag() + b)
P = -1j*np.sqrt(2)*(b.dag() - b)

for n in range(len(N_traj)):
    Xprom = np.mean(
            [
                np.array([
                    expect(X, phi)
                    for phi in tray
                ])
                for tray in tray_n[n]
            ],
            axis=0
        )
    Pprom = np.mean(
            [
                np.array([
                    expect(P, phi)
                    for phi in tray
                ])
                for tray in tray_n[n]
            ],
            axis=0
        )

import matplotlib.pyplot as plt


plt.title('Espacio fase')
plt.grid(0.5)
plt.xlabel(r'$\langle X \rangle$')
plt.ylabel(r'$\langle P \rangle$')
plt.scatter(Pprom,Xprom, s= 1, c='darkred')

In [ ]:
from qutip import *
n=50
a= destroy(n)
delta = 1
k = 1/20
e = 1.5


H_s = omega_0 * (a.dag() *a + 1/2) + (0.2*np.sqrt(2)*(a.dag() + a)).sinm() + np.sqrt(2)*(a.dag() + a) 
H_k = - delta * (a.dag()* a) + k*0.5* (a.dag()*a.dag()*a*a) + e * 0.25 * (a.dag()*a.dag() + a*a)

eigs = H_k.eigenenergies()
print(eigs)
plt.figure(figsize=(6, 6))
plt.hlines(y=eigs, xmin=-5, xmax=5, colors='red', linewidth=1, alpha=0.7)

plt.title('Niveles de energía de H')
plt.ylabel('Eigenvalores de H  ')

plt.xticks([])
plt.xlim(-7, 7)
plt.ylim(eigs.min() - 1, eigs.max() + 1)
plt.grid(axis='y', linestyle='--', alpha=0.5) # Cuadrícula solo en el eje Y
plt.show()


# Oscilador amortiguado por un baño térmico a temp T.
El ejemplo anterior era el caso para un sistema de 2 niveles o de un oscilador armonico con un acoplamiento tipo emision espontanea, i.e., un acoplamiento a un baño de osciladores en su estado base (T = 0), para un reservorio con temperatura T el operador de relajacion se vuelve

$$
\mathcal L_{rlx}(\rho) = \frac{\Gamma}{2}[1 + n(\omega_0)](\sigma^-\rho\sigma^+ -\frac{1}{2}\{\sigma^+\sigma^-, \rho\}) + \frac{\Gamma}{2}[n(\omega_0)](\sigma^+\rho\sigma^- -\frac{1}{2}\{\sigma^-\sigma^+, \rho\})
$$
Con operadores de salto, que corresponden a una perdida por emision espontanea o por una emision estimulada ($L_1$) y una excitacion (ganancia) por absorción del reservorio ($L_2$)
$$
L_k = \left\{\sqrt{\Gamma[1+ n(\omega_0)]}\sigma^- , \sqrt{\Gamma n(\omega_0)}\sigma^+\right\}
$$
Donde $n(\omega_0)$ es la media de 'excited quanta' a temperatura T a la frecuencia de resonancia $\omega_0$ del sistema de 2 niveles o el oscilador

$$
n(\omega_0) = \left[\exp{\frac{\hbar\omega_0}{kT}-1}\right]^{-1}
$$
Tomando en cuenta que $\sigma^- =\rightarrow b$ Para un oscilador amortiguado por un baño térmico.

In [ ]:
"""
Reservorio a una temperatura T: Oscilador amortiguado por un baño térmico.
"""

N = 50 # Espacio de Hilbert
ket0 = basis(N, 0) 
ket1 = basis(N,1)

# Parametros 
alpha = 0.35 - 0.2j
if np.abs(alpha) > 1:
    print('el modulo de alpha debe ser menor a 1')
    sys.exit()
else:
    beta = np.sqrt(complex(1 - alpha * alpha.conjugate()))
Gamma = 1 
omega_0 = 0.5
T = 1000
# Estado inicial
phi_0 = alpha*ket0 + beta*ket1


b = destroy(N)
I = qeye(N)
H_s = omega_0 * (b.dag() * b)

n_omega = (np.exp(omega_0/T))**(-1)

L_ops = [
        np.sqrt(Gamma*(1+ n_omega)) * b,
        np.sqrt(Gamma*n_omega) * b.dag()
        ]

N_traj = [40, 60,80,120]

tray_n = []

for N in N_traj:

    trayectorias = []
    
    for i in range(N):

        trayectoria = MCWF_evolution(
            phi_0,
            H_s,
            L_ops,
            t_final=10,
            dt=0.005
        )

        trayectorias.append(trayectoria)

    tray_n.append(np.array(trayectorias))

print('finished')

In [ ]:
"""
Evolucion exacta de la ecuacion de Linblad para comparar
"""
t = np.arange(0, 10 , 0.005)
rho_0 = phi_0*phi_0.dag()
n_op = [b.dag()*b]
evo = mesolve(H_s , rho_0, t, L_ops, e_ops = n_op)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for n in range(len(N_traj)):

    promedio = np.mean(
        [
            np.array([
                expect(n_op, phi)
                for phi in tray
            ])
            for tray in tray_n[n]
        ],
        axis=0
    )

    plt.plot(
        t,
        promedio,
        label=fr"$N={N_traj[n]}$"
    )
plt.plot(t, evo.expect[0], c='black', label = 'Linblad eq.')

#plt.ylim(0.6,1.5)
plt.xlabel(r"$t$")
plt.ylabel(r"$\langle n \rangle$")
plt.legend()
plt.grid()

plt.savefig('DampedHO_TReservoir.pdf')
plt.show()

# Reservorio con temp finita: Atomo de 2 niveles


In [ ]:
"""
Reservorio a una temperatura T: Atomo de 2 niveles
"""

N = 2 # Espacio de Hilbert
ket0 = basis(N, 0) # up
ket1 = basis(N, 1) # down

# Parametros 
alpha = 0.35 - 0.2j
if np.abs(alpha) > 1:
    print('el modulo de alpha debe ser menor a 1')
    sys.exit()
else:
    beta = np.sqrt(complex(1 - alpha * alpha.conjugate()))
    
Gamma = 1 
omega_0 = 0.5
T = 500
# Estado inicial
phi_0 = spin_coherent(1/2, np.pi/4, np.pi/4)

# Operadores de spin
sig_ = sigmam()
I = qeye(N)
H_s = omega_0 * (sig_.dag() * sig_)

n_omega = (np.exp(omega_0/T))**(-1)

L_ops = [
        np.sqrt(Gamma*(1+ n_omega)) * sig_ * 0.1,
        np.sqrt(Gamma*n_omega) * sig_.dag()
        ]

N_traj = [40, 60,80,120]

tray_n = []

for N in N_traj:

    trayectorias = []
    
    for i in range(N):

        trayectoria = MCWF_evolution(
            phi_0,
            H_s,
            L_ops,
            t_final=10,
            dt=0.05
        )

        trayectorias.append(trayectoria)

    tray_n.append(np.array(trayectorias))

print('----------------------------finished MCWF----------------------------')



In [ ]:
"""
Evolucion exacta de la ecuacion de Linblad para comparar
"""
t = np.arange(0, 10 , 0.05)
rho_0 = phi_0*phi_0.dag()



exp_ops = [jmat(1/2, 'x'),jmat(1/2, 'y'),jmat(1/2, 'z'),
           jmat(1/2, 'x')@jmat(1/2, 'x'),
           jmat(1/2, 'y')@jmat(1/2, 'y'),
           jmat(1/2, 'z')@jmat(1/2, 'z')]
evo = mesolve(H_s , rho_0, t, L_ops, e_ops=exp_ops)

print('----------------------------finished integrating----------------------------')
#print(evo.expect)

In [ ]:
"""
Evolucion exacta de la ecuacion de Linblad para comparar
"""
Gamma = 1
omega_0 = 0.5
T = 500
# Estado inicial
phi_0 = spin_coherent(1/2, np.pi/4, np.pi/4)

# Operadores de spin
sig_ = sigmam()
I = qeye(N)
H_s = omega_0 * (sig_.dag() * sig_) + 10*sigmax()
t = np.arange(0, 10 , 0.005)
rho_0 = phi_0*phi_0.dag()
L_ops = [
        np.sqrt(Gamma*(1+ n_omega)) * sig_ ,
        np.sqrt(Gamma*n_omega) * sig_.dag(),
        100 * sigmaz()
        ]


exp_ops = [jmat(1/2, 'x'),jmat(1/2, 'y'),jmat(1/2, 'z'),
           jmat(1/2, 'x')@jmat(1/2, 'x'),
           jmat(1/2, 'y')@jmat(1/2, 'y'),
           jmat(1/2, 'z')@jmat(1/2, 'z')]
#evo = mesolve(H_s , rho_0, t, L_ops, e_ops=exp_ops)
st = mesolve(H_s , rho_0, t, L_ops)
print('----------------------------finished integrating----------------------------')
#print(evo.expect)

plt.plot(t, evo.expect[2], c='black', label = 'Linblad eq.', alpha=0.5)

#plt.ylim(0.6,1.5)
plt.xlabel(r"$t$")
plt.ylabel(r"$\langle S^z \rangle$")
plt.legend()
plt.grid()

plt.show()


print(st.states[2])

In [ ]:
plt.figure(figsize=(8, 5))

for n in range(len(N_traj)):

    promedio = np.mean(
        [
            np.array([
                expect(exp_ops[2], phi)
                for phi in tray
            ])
            for tray in tray_n[n]
        ],
        axis=0
    )

    plt.plot(
        t,
        promedio,
        label=fr"$N={N_traj[n]}$"
    )
plt.plot(t, evo.expect[2], c='black', label = 'Linblad eq.', alpha=0.5)

#plt.ylim(0.6,1.5)
plt.xlabel(r"$t$")
plt.ylabel(r"$\langle S^z \rangle$")
plt.legend()
plt.grid()

plt.savefig('2lvlAtom.pdf')
plt.show()

# Esfera de Poincare

Estado coherente de spin se construye a partir de
$$
|\phi ,\theta \rangle =  e^{-i\phi J_z} e^{-i\theta J_y} | j, j \rangle
$$
Calculando los valores esperados de cada op. de spin se obtiene
$$
\langle J_x\rangle = j \sin \theta \cos\phi 
$$
$$
\langle J_y\rangle = j \sin \theta \sin \phi 
$$
$$
\langle J_z\rangle = j \cos \theta
$$
Se nota que estos valores son similares a la transformacion de coordenadas cartesianas a esfericas, con j siendo el radio de la esfera (momento angular total)
$$
\Delta X = \langle X^2 \rangle - \langle X \rangle^2
$$

In [ ]:
Sx = evo.expect[0]
Sy = evo.expect[1]
Sz = evo.expect[2]

Sx2 = evo.expect[3]
Sy2 = evo.expect[4]
Sz2 = evo.expect[5]
j=1/2
S2 = Sx*Sx + Sy*Sy + Sz*Sz
print(min(Sx))
print(min(Sy))
print(min(Sz))

dSx = Sx2 - Sx*Sx
dSy = Sy2 - Sy*Sy
dSz = Sz2 - Sz*Sz
print(min(dSx))
print(min(dSy))
print(min(dSz))

phiq   = np.atan2(Sy,Sx)
thetaq = np.atan2(np.sqrt(Sx*Sx + Sy*Sy),Sz) 

Sx_prom = np.mean(
        [
            np.array([
                expect(exp_ops[0], phi)
                for phi in tray
            ])
            for tray in tray_n[3]
        ],
        axis=0
    )
Sy_prom = np.mean(
        [
            np.array([
                expect(exp_ops[1], phi)
                for phi in tray
            ])
            for tray in tray_n[3]
        ],
        axis=0
    )
Sz_prom = np.mean(
        [
            np.array([
                expect(exp_ops[2], phi)
                for phi in tray
            ])
            for tray in tray_n[3]
        ],
        axis=0
    )
print('\n' + 'MCWF')
print(min(Sx_prom))
print(min(Sy_prom))
print(min(Sz_prom))

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')

# ============================================================
# Esfera de estados coherentes
# ============================================================

theta_vals = np.linspace(0, np.pi, 20)
phi_vals = np.linspace(0, 2*np.pi, 40)

Jx_vals = []
Jy_vals = []
Jz_vals = []

j = 1/2

for theta in theta_vals:
    for phi in phi_vals:

        psi = spin_coherent(j, theta, phi)

        Jx = expect(jmat(j, 'x'), psi)
        Jy = expect(jmat(j, 'y'), psi)
        Jz = expect(jmat(j, 'z'), psi)

        Jx_vals.append(Jx)
        Jy_vals.append(Jy)
        Jz_vals.append(Jz)


# ============================================================
# Trayectoria de Lindblad
# ============================================================

phiq = np.atan2(Sy, Sx)

thetaq = np.atan2(
    np.sqrt(Sx*Sx + Sy*Sy),
    Sz
)

x = j * np.sin(thetaq) * np.cos(phiq)
y = j * np.sin(thetaq) * np.sin(phiq)
z = j * np.cos(thetaq)

x = Sx_prom
y = Sy_prom
z = Sz_prom
# ============================================================
# Esfera de Poincaré / Bloch
# ============================================================

scatter = ax.scatter(
    Jx_vals,
    Jy_vals,
    Jz_vals,
    c=Jz_vals,
    cmap='viridis',
    s=10,
    alpha=0.8
)

fig.colorbar(
    scatter,
    ax=ax,
    shrink=0.5,
    aspect=10,
    label=r'$J_z$'
)


# ============================================================
# Configuración
# ============================================================

ax.set_title(
    f"Esfera de Poincaré para spin j = {float(j):.2f}"
)

ax.set_xlabel(r"$J_x$")
ax.set_ylabel(r"$J_y$")
ax.set_zlabel(r"$J_z$")

ax.set_box_aspect([1, 1, 1])

# Límites para que la esfera no cambie de escala
ax.set_xlim(-j, j)
ax.set_ylim(-j, j)
ax.set_zlim(-j, j)

# ============================================================
# Objetos que se van a animar
# ============================================================

line, = ax.plot([], [], [], lw=2)

point, = ax.plot(
    [], [], [],
    marker='o',
    markersize=8,
    linestyle=''
)


# ============================================================
# Función de actualización
# ============================================================

def update(frame):

    # Trayectoria desde t=0 hasta t[frame]
    line.set_data(
        x[:frame+1],
        y[:frame+1]
    )

    line.set_3d_properties(
        z[:frame+1]
    )

    # Punto actual
    point.set_data(
        [x[frame]],
        [y[frame]]
    )

    point.set_3d_properties(
        [z[frame]]
    )

    return line, point


# ============================================================
# Crear animación
# ============================================================

ani = FuncAnimation(
    fig,
    update,
    frames=range(0, len(t), 10),
    interval=30,
    blit=False
)


# ============================================================
# Mostrar en Jupyter
# ============================================================

plt.close(fig)

HTML(ani.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Crear puntos sobre una esfera de radio 5
n_puntos = 5
theta = np.linspace(0, 2*np.pi, n_puntos)
phi = np.linspace(0, np.pi, n_puntos)

x = 5 * np.sin(phi) * np.cos(theta)
y = 5 * np.sin(phi) * np.sin(theta)
z = 5 * np.cos(phi)

# 2. Definir incertidumbres (dx, dy, dz) para cada punto
dx = np.random.uniform(0.3, 0.8, size=n_puntos)
dy = np.random.uniform(0.3, 0.8, size=n_puntos)
dz = np.random.uniform(0.3, 0.8, size=n_puntos)

# Configurar gráfico 3D
fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=(8, 8))

# Dibujar los puntos centrales reales
ax.scatter(x, y, z, color='red', s=50, label='Puntos medidos', zorder=5)

# 3. Generar y dibujar un elipsoide para cada punto
u = np.linspace(0, 2 * np.pi, 20)
v = np.linspace(0, np.pi, 20)

for i in range(n_puntos):
    # Ecuación paramétrica del elipsoide escalado por dx, dy, dz y centrado en (x,y,z)
    x_elip = x[i] + dx[i] * np.outer(np.cos(u), np.sin(v))
    y_elip = y[i] + dy[i] * np.outer(np.sin(u), np.sin(v))
    z_elip = z[i] + dz[i] * np.outer(np.ones_like(u), np.cos(v))
    
    # Dibujar la superficie del elipsoide con transparencia
    ax.plot_surface(x_elip, y_elip, z_elip, color='blue', alpha=0.2, edgecolor='none')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Puntos en Esfera con Elipsoides de Incertidumbre (dx, dy, dz)')
plt.legend()
plt.show()


In [ ]:
print(len(t))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import qutip as qt

# ==========================================
# 1. YOUR EXISTING DATA (Replace these arrays with yours)
# ==========================================
# Example: Let's assume you already have these arrays calculated


x_points =  np.sin(thetaq)* np.cos(phiq)
y_points =  np.sin(thetaq)* np.sin(phiq)
z_points =  np.cos(thetaq)

x_points =  Sx
y_points =  Sy
z_points =  Sz
# ==========================================

# 2. Setup the Bloch sphere figure
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
b = qt.Bloch(fig=fig, axes=ax)

# 3. Create the update logic using your array indices
def update(frame):
    b.clear()   # Wipe old points from data queue
    ax.clear()  # Clear canvas to avoid double lines
    
    # Re-draw the clean Bloch sphere backdrop
    b.make_sphere() 
    
    # Pull the exact x, y, z for this specific frame
    x = x_points[frame]
    y = y_points[frame]
    z = z_points[frame]
    
    # Optional: If you want to show a permanent trail of where the point has been,
    # uncomment the two lines below:
    # trail_x, trail_y, trail_z = x_points[:frame+1], y_points[:frame+1], z_points[:frame+1]
    # b.add_points([trail_x, trail_y, trail_z], meth='l') # 'l' makes it a line trail
    
    # Add the current single temporary point
    b.add_points([x, y, z])
    b.render()
    
    return ax.artists

# 4. Generate and display the native flicker-free video
# interval=40 means 25 frames per second (1000ms / 40ms)
ani = FuncAnimation(fig, update, frames=range(0, len(t), 10), interval=30, blit=False)
plt.close() 

# Render inside Colab
HTML(ani.to_html5_video())

In [ ]:
from qutip import *
from scipy import *
def qubit_integrate(w, theta, gamma1, gamma2, psi0, tlist):
    # operators and the hamiltonian
    sx = sigmax(); sy = sigmay(); sz = sigmaz(); sm = sigmam()
    H = w * (np.cos(theta) * sz + np.sin(theta) * sx)
    # collapse operators
    c_op_list = []
    n_th = 0.5 # temperature
    rate = gamma1 * (n_th + 1)
    if rate > 0.0: c_op_list.append(np.sqrt(rate) * sm)
    rate = gamma1 * n_th
    if rate > 0.0: c_op_list.append(np.sqrt(rate) * sm.dag())
    rate = gamma2
    if rate > 0.0: c_op_list.append(np.sqrt(rate) * sz)


    # evolve and calculate expectation values
    output = mesolve(H, psi0, tlist, c_op_list, e_ops=[sx, sy, sz])  
    return output.expect[0], output.expect[1], output.expect[2]
    
## calculate the dynamics
w     = 1.0 * 2 * np.pi   # qubit angular frequency
theta = 0.2 * np.pi       # qubit angle from sigma_z axis (toward sigma_x axis)
gamma1 = 0.5      # qubit relaxation rate
gamma2 = 0.2      # qubit dephasing rate
# initial state
a = 1.0
psi0 = (a* basis(2,0) + (1-a)*basis(2,1))/(np.sqrt(a**2 + (1-a)**2))
tlist = np.linspace(0,4,250)
#expectation values for ploting
sx, sy, sz = qubit_integrate(w, theta, gamma1, gamma2, psi0, tlist)

